# Harvest Campaign Catalog for CIDs
https://radiantearth.github.io/stac-browser/#/external/ipfs.orcestra-campaign.org/ipfs/bafybeihihawyfoolrnrrvyujuefs7qh7mifcvjaub3guiug7ucx47pokhy/catalog.json

In [1]:
import requests

# Root STAC catalog URL
catalog_url = "https://ipfs.orcestra-campaign.org/ipfs/bafybeihihawyfoolrnrrvyujuefs7qh7mifcvjaub3guiug7ucx47pokhy/catalog.json"

def extract_zarr_urls(catalog_url):
    catalog = requests.get(catalog_url).json()
    zarr_list = []

    # Iterate over each item link in the catalog
    for link in catalog.get("links", []):
        if link.get("rel") == "item":
            item_url = catalog_url.rsplit("/", 1)[0] + "/" + link["href"]
            item = requests.get(item_url).json()

            # Look for assets with type or href pointing to Zarr
            for asset_name, asset in item.get("assets", {}).items():
                href = asset.get("href", "")
                if href.endswith(".zarr") or "zarr" in asset.get("type", "").lower():
                    
                    zarr_list.append({
                        "dataset_name": item.get("properties", {})["title"],
                        "zarr_url": href
                    })
    return zarr_list

if __name__ == "__main__":
    zarr_datasets = extract_zarr_urls(catalog_url)
    for ds in zarr_datasets:
        print(f"{ds['dataset_name']}: {ds['zarr_url']}")

MAESTRO-2024 ATR42 Chemistry Observation - Ozone: ipfs://bafybeigkp57oro2mx6pgbtqzajjghvhypg33btkd5sicf6zegvvptu7yda
MAESTRO-2024 ATR42 Navigation parameters: ipfs://bafybeifhj7cdxa5edqhgn2fvfa3ticiy6pz2f46jxye7r4hnihawzbxjra
MAESTRO-2024 ATR42 Thermodynamics and Dynamic Core Observation: ipfs://bafybeidtrcpmrymqfep6caqc3xe4hneby3xsgpsy37pghk4rmbbdggg6ze
MAESTRO-2024 ATR42 Temperature and Humidity Fast Observation: ipfs://bafybeiapsnej3k5q374wynbox77zs537afyvnqoubxh5yfvwehvhwicfam
MAESTRO-2024 ATR42 Liquid Water Content measurements: ipfs://bafybeiayh6lqzcxm7nvegmjqsqhdgww7mtzs2m3uskjmyqobeqqrlefrs4
MAESTRO-2024 ATR42 Radiation Observations: ipfs://bafybeiauxz4axsdbal7qiqcpkehcyafjs7zk2pltqadttvmsdh3q2xis5y
MAESTRO-2024 ATR42 SPP300 parameters: ipfs://bafybeihzcwakly2vlkifrdktcte3njleh6gouvn5lieyxdp4irpap2gf3i
Broadband solar and terrestrial, upward and downward irradiance measured by BACARDI on HALO during the PERCUSION field campaign: ipfs://bafybeiaoalflfftmsfqakenwp5gpxnxfjhaxkrjq7

# Datasets which caused Errors

Continuous subskin sea surface temperature data along RV METEOR M203 cruise track

ipfs://bafybeieoosfm33u47expejxhccectau4oxrp2jg4efzrqn743af6tzxide

---
Wind LiDAR LiTra S raw data (ship motion present)

ipfs://bafybeibg3e7kodnqrgjs5576soeq7gwcqjzdity56zlmz2bw27ucrip4fe

---
Wind LiDAR LiTra S heave-corrected, synchronous ship motion data

ipfs://bafybeigh4aqyy3zmhqteq4y53zknarkmy7rwe6sctcwcskuyw4paap6kxm

---
Wind LiDAR LiTra S heave-corrected, asynchronous ship motion data

ipfs://bafybeiaonfwwrbianryo4nejd4fyfqjcrofo6wciz62aeuzpeplcqu6zqe

In [2]:
# Exclude erroneous datasets from writing
zarr_datasets_remainder = zarr_datasets[49::]
zarr_datasets_remainder

[{'dataset_name': 'Ceilometer (CHM15k Nimbus) measurements during METEOR cruise M203',
  'zarr_url': 'ipfs://bafybeifpycgdtmphy2kbvo7a2tchelswunmyiqhrc7qiqevubjon7bbfcm'},
 {'dataset_name': 'Cloud radar and Cloudnet on RV Meteor during BOWTIE',
  'zarr_url': 'ipfs://bafybeiakgiqypaykbbyxxaesqxz2g4kjwjactcpvju2tjutztm5hpxl2km'},
 {'dataset_name': 'METEOR1 PARSIVEL Disdrometer QCed data',
  'zarr_url': 'ipfs://bafybeicrrokylncpyjgs52c6526otqbqxcxhmkt6mkqovya2xvt2n3yo54'},
 {'dataset_name': 'METEOR2 PARSIVEL Disdrometer QCed data',
  'zarr_url': 'ipfs://bafybeiacc53awulhrttttkdwd6uktvb3bfbsyac5g4ww7rkujigkalmkfe'},
 {'dataset_name': 'Merged PARSIVEL Disdrometer QCed data',
  'zarr_url': 'ipfs://bafybeihjcwsecgpmsjxoo5peqafnuqfnalu3ya3vtibwl7qkm76izsnuei'},
 {'dataset_name': 'surface rain flag from Micro Rain Radar',
  'zarr_url': 'ipfs://bafybeie2zhy2rkepcqh6tc5lk75blv372rvpfnnrhsyl6efscdgmugld4a'},
 {'dataset_name': 'Rain gauge measurements during METEOR cruise M203',
  'zarr_url': 'ipfs

# Write to Objectstore

In [3]:
import xarray as xr
import s3fs
import re

# -----------------------------
# Object storage configuration
# -----------------------------
S3_ENDPOINT = "https://s3.eu-dkrz-3.dkrz.cloud"
BUCKET_NAME = "orchestra"

# -----------------------------
# Connect to object storage
# -----------------------------
fs = s3fs.S3FileSystem(
    profile="default",
    client_kwargs={"endpoint_url": S3_ENDPOINT},
)

def write_datasets_to_s3(zarr_list):
    """
    Opens datasets from URIs and writes them chunk-by-chunk to S3

    """
    
    for dataset in zarr_list:
        print(f"Processing dataset: {dataset['zarr_url']}")
        
        # Open dataset lazily with automatic chunking
        ds = xr.open_dataset(f"{dataset['zarr_url']}", engine="zarr")#, chunks="auto") # Chunks Auto causes problems

        
        filename = f"{dataset['dataset_name']}"
        zarr_path = re.sub(r"[^\w\-]+", "_", filename) + ".zarr"
        #zarr_path = f"{dataset['dataset_name']}".replace(" ", "_").replace("/", "_").replace(",", "") +".zarr"
        # Map S3 location
        store = s3fs.S3Map(
            root=f"{BUCKET_NAME}/{zarr_path}",
            s3=fs,
            check=False
        )
        
        # Write lazily chunk-by-chunk
        ds.to_zarr(
            store,
            mode="w",
            zarr_format=2,
            consolidated=True,
        )
        
        print(f"Dataset successfully written to: s3://{BUCKET_NAME}/{zarr_path}")


if __name__ == "__main__":
    write_datasets_to_s3(zarr_datasets_remainder)

Processing dataset: ipfs://bafybeifpycgdtmphy2kbvo7a2tchelswunmyiqhrc7qiqevubjon7bbfcm
Dataset successfully written to: s3://orchestra/Ceilometer_CHM15k_Nimbus_measurements_during_METEOR_cruise_M203.zarr
Processing dataset: ipfs://bafybeiakgiqypaykbbyxxaesqxz2g4kjwjactcpvju2tjutztm5hpxl2km
Dataset successfully written to: s3://orchestra/Cloud_radar_and_Cloudnet_on_RV_Meteor_during_BOWTIE.zarr
Processing dataset: ipfs://bafybeicrrokylncpyjgs52c6526otqbqxcxhmkt6mkqovya2xvt2n3yo54
Dataset successfully written to: s3://orchestra/METEOR1_PARSIVEL_Disdrometer_QCed_data.zarr
Processing dataset: ipfs://bafybeiacc53awulhrttttkdwd6uktvb3bfbsyac5g4ww7rkujigkalmkfe
Dataset successfully written to: s3://orchestra/METEOR2_PARSIVEL_Disdrometer_QCed_data.zarr
Processing dataset: ipfs://bafybeihjcwsecgpmsjxoo5peqafnuqfnalu3ya3vtibwl7qkm76izsnuei
Dataset successfully written to: s3://orchestra/Merged_PARSIVEL_Disdrometer_QCed_data.zarr
Processing dataset: ipfs://bafybeie2zhy2rkepcqh6tc5lk75blv372rvpfnnr